# Fig. 7b: information capacity of the ground truth and of the sorted populations (evoked model)

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel b of the paper: manifold information capacity, divided by its lower bound, of the ground truth neurons, of the sorted single-units of each spike sorter and of the multiunit threshold crossing. Source data and svg are saved in figures/8_source_data/fig7b/.

**method (as in the paper's notebook fig7_B_info_capacity_by_sorter_revised.ipynb)**
* the capacity of each population is computed on 200 bootstraps of the neurons or units (the population responses to the 360 stimulus directions, reduced to 200 latents, `neural_manifolds_replicaMFT`); the capacity of a population divided by the lower bound capacity obtained by shuffling the mapping between responses and stimulus directions is 1 at chance
* the 200 bootstrapped capacities of each population are read from the csv files of the paper (`igeometrics_by_sorter.csv`, `igeometrics_thresh_crossing.csv`); they are **not recomputed** here (the capacity analysis needs the `mftma` package)
* box plots (notch, 200 bootstraps); red dashed line: mean capacity of the ground truth; blue dashed line: chance (1)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import spikeinterface as si
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# set project path
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(PROJ_PATH)

# custom package
import seaborn as sns
from scipy.stats import ttest_ind

# paths (bootstrapped manifold capacities of the ground truth and of the sorted populations, evoked simulation, 1 hour)
NEURAL_CODE_PATH = "dataset/01_intermediate/neural_code/npx_evoked/"
IGEOM_SORT = NEURAL_CODE_PATH + "igeometrics_by_sorter.csv"
IGEOM_TC = NEURAL_CODE_PATH + "igeometrics_thresh_crossing.csv"

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1
legend_cfg = {"frameon": False, "handletextpad": 0.5}

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
# Nature figure guidelines for every saved figure: Arial, 5-7 pt, editable text, images of at least 300 dpi, a .pdf next to each .svg
import os, sys
sys.path.insert(0, os.getcwd())     # the repository (the notebook moved there above)
from src.nodes import figure_style
figure_style.apply()

## Load the bootstrapped capacities

In [2]:
%%time

igeom_df = pd.read_csv(IGEOM_SORT)
igeom_tc_df = pd.read_csv(IGEOM_TC)
igeom_df = pd.concat([igeom_df, igeom_tc_df[igeom_tc_df["Sorter"] == "thresh-crossing"]], axis=0, ignore_index=True)

# populations in the order of the figure
NAMES = {"ground_truth": "Ground\ntruth", "KS4": "KS4", "KS3": "KS3", "KS2.5": "KS2.5", "KS2": "KS2", "KS": "KS", "HS": "HS", "thresh-crossing": "TC"}
df = igeom_df[igeom_df["Sorter"].isin(NAMES)][["Sorter", "Capacity"]].copy()
df["population"] = df["Sorter"].map(NAMES)
df["bootstrap"] = df.groupby("Sorter").cumcount()
df = df.set_index("Sorter").loc[list(NAMES)].reset_index(drop=True)[["population", "bootstrap", "Capacity"]]
df = df.rename(columns={"Capacity": "capacity_over_lower_bound"})
print(df.groupby("population", sort=False)["capacity_over_lower_bound"].agg(["count", "median"]).round(3))

               count  median
population                  
Ground\ntruth    200   1.249
KS4              200   1.148
KS3              200   1.122
KS2.5            200   1.171
KS2              200   1.141
KS               200   1.037
HS               200   1.132
TC               200   1.223
CPU times: user 5.83 ms, sys: 971 µs, total: 6.8 ms
Wall time: 6.77 ms


## Capacity of each sorted population against the ground truth

In [3]:
gt = df[df["population"] == "Ground\ntruth"]["capacity_over_lower_bound"]
rows = []
for name in list(NAMES.values())[1:]:
    c = df[df["population"] == name]["capacity_over_lower_bound"]
    t, p = ttest_ind(gt, c)
    rows.append(dict(population=name.replace("\n", " "), median_capacity=c.median(), reduction_of_capacity_above_lower_bound_percent=100 * (gt.median() - c.median()) / (gt.median() - 1), t_statistic=t, p_value=p))
stats = pd.DataFrame(rows)
display(stats.round(4))
print("ground truth median capacity:", round(gt.median(), 3), "| average reduction over the 6 spike sorters (%):", round(stats["reduction_of_capacity_above_lower_bound_percent"].iloc[:6].mean(), 1))

,population,median_capacity,reduction_of_capacity_above_lower_bound_percent,t_statistic,p_value
0,KS4,1.1478,40.5712,30.9221,0.0
1,KS3,1.1220,50.9546,38.0045,0.0
2,KS2.5,1.1710,31.2338,24.0776,0.0
3,KS2,1.1414,43.1673,33.7892,0.0
4,KS,1.0372,85.0299,68.0796,0.0
5,HS,1.1318,46.9883,36.8614,0.0
6,TC,1.2228,10.4346,8.3778,0.0


ground truth median capacity: 1.249 | average reduction over the 6 spike sorters (%): 49.7


## Plot, same as the paper's panel b

In [4]:
# paper geometry (page 22 of the paper): canvas 160.5 x 134.5 pt, Arial 5 pt text, 7 pt title, editable (src/nodes/validation/capacity_plots.py)
from src.nodes.validation.capacity_plots import plot_information_capacity

def plot_source_data(df: pd.DataFrame):
    return plot_information_capacity(df)

## Plot and save the figure and the source data

In [5]:
%%time

# plot
fig = plot_source_data(df)

# save figure
os.makedirs("figures/8_source_data/fig7b", exist_ok=True)
plt.savefig("figures/8_source_data/fig7b/fig7b.svg", transparent=True)   # the .pdf is written next to the .svg

# save source data (sheet 1: capacity of the 200 bootstraps of each population; sheet 2: comparison with the ground truth)
with pd.ExcelWriter("figures/8_source_data/fig7b/fig7b.xlsx") as writer:
    df.to_excel(writer, sheet_name="capacity_bootstraps", index=False)
    stats.to_excel(writer, sheet_name="vs_ground_truth", index=False)

## Check: replot from the saved source data

In [6]:
%%time

df = pd.read_excel("figures/8_source_data/fig7b/fig7b.xlsx", sheet_name="capacity_bootstraps")
fig = plot_source_data(df)